<a href="https://colab.research.google.com/github/R-Broco/naporta-api-do-rafa/blob/main/pipeline_dashboard_bi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install psycopg2-binary pandas sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 43.2 MB/s eta 0:00:00


In [3]:
import os
import pandas as pd
from google.colab import userdata
from sqlalchemy import create_engine, text

# 1. Recupera a URL de forma segura da aba Secrets do Colab
try:
    db_url = userdata.get('DATABASE_URL')
    # Ajuste técnico: O SQLAlchemy moderno exige que a string comece com 'postgresql://'
    # e não 'postgres://' (caso sua string antiga use postgres).
    if db_url.startswith("postgres://"):
        db_url = db_url.replace("postgres://", "postgresql://", 1)
except Exception as e:
    print("❌ Erro: Certifique-se de que ativou o acesso à chave 'DATABASE_URL' no menu de Secrets (ícone de chave).")
    raise e

# 2. Cria o motor de conexão (Engine) do SQLAlchemy
engine = create_engine(db_url)

# 3. Teste rápido de conexão executando uma query simples
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT version();"))
        version = result.fetchone()
        print("✅ Conexão com o banco PostgreSQL (Neon) estabelecida com sucesso!")
        print(f"📊 Versão do Banco: {version[0]}")
except Exception as e:
    print("❌ Falha na conexão. Verifique as credenciais ou parâmetros de SSL na string.")
    print(f"Erro original: {e}")


✅ Conexão com o banco PostgreSQL (Neon) estabelecida com sucesso!
📊 Versão do Banco: PostgreSQL 18.4 (48c2093) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [5]:
# Script rápido para descobrir os nomes reais das colunas no banco
with engine.connect() as conn:
    # Lista as colunas da tabela Pedido
    res_p = conn.execute(text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = 'Pedido';
    """))
    print("📋 Colunas reais na tabela Pedido:")
    for row in res_p:
        print(f" - {row[0]}")

    # Lista as colunas da tabela Item
    res_i = conn.execute(text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = 'Item';
    """))
    print("\n📋 Colunas reais na tabela Item:")
    for row in res_i:
        print(f" - {row[0]}")


📋 Colunas reais na tabela Pedido:
 - id
 - numero
 - dataPrevisaoEntrega
 - clienteNome
 - clienteDocumento
 - enderecoEntrega
 - status
 - createdAt
 - updatedAt
 - deletedAt

📋 Colunas reais na tabela Item:
 - id
 - descricao
 - preco
 - pedidoId


In [6]:
import pandas as pd

# 1. Query SQL calibrada milimetricamente com o seu Schema do Prisma e as aspas do Postgres
query = """
SELECT
    p.id AS pedido_id,
    p.numero AS numero_pedido,
    p."dataPrevisaoEntrega" AS data_previsao,
    p."clienteNome" AS cliente_nome,
    p."clienteDocumento" AS cliente_documento,
    p."enderecoEntrega" AS endereco_entrega,
    p.status AS status_pedido,
    p."createdAt" AS data_criacao,
    i.id AS item_id,
    i.descricao AS item_descricao,
    i.preco AS item_preco
FROM "Pedido" p
INNER JOIN "Item" i ON p.id = i."pedidoId"
WHERE p."deletedAt" IS NULL;
"""

# 2. Carrega os dados salvando direto no DataFrame do Pandas
try:
    df_raw = pd.read_sql(query, con=engine)
    print(f"📊 Extração concluída! {len(df_raw)} registros de itens encontrados no banco.")

    if not df_raw.empty:
        # 3. Conversão de Precisão: Transforma o Decimal(10,2) do Postgres para Float do Pandas
        df_raw['item_preco'] = df_raw['item_preco'].astype(float)
        print("✅ Preços dos itens convertidos com sucesso para Float.")

        # 4. Agrupamento (Group By) para somar os valores dos itens por pedido
        df_pedidos = df_raw.groupby([
            'pedido_id', 'numero_pedido', 'data_previsao',
            'cliente_nome', 'cliente_documento', 'endereco_entrega',
            'status_pedido', 'data_criacao'
        ]).agg(
            valor_total=('item_preco', 'sum'),
            total_itens=('item_id', 'count')
        ).reset_index()

        print(f"📦 Agrupamento concluído! {len(df_pedidos)} pedidos únicos gerados.")

        # 5. Exibe a tabelinha de prévia dos dados consolidados na tela
        print("\n👀 Veja a prévia dos pedidos processados abaixo:")
        display(df_pedidos.head())
    else:
        print("⚠️ Atenção: O banco de dados está vazio ou todos os registros ativos foram excluídos logicamente.")

except Exception as e:
    print("❌ Erro durante o processamento:")
    print(f"Detalhes: {e}")


📊 Extração concluída! 5 registros de itens encontrados no banco.
✅ Preços dos itens convertidos com sucesso para Float.
📦 Agrupamento concluído! 3 pedidos únicos gerados.

👀 Veja a prévia dos pedidos processados abaixo:


,pedido_id,numero_pedido,data_previsao,cliente_nome,cliente_documento,endereco_entrega,status_pedido,data_criacao,valor_total,total_itens
0,18b81ef0-a552-419e-a03a-0c19c337974f,PED-1003,2026-06-15 09:00:00,Carlos Almeida,99988877766,"Rua do Ouvidor, 50, Centro, Rio de Janeiro - RJ",PENDENTE,2026-06-15 13:31:57.323,645.99,2
1,66e6bdf3-337f-4ab1-ab5a-6c074f4e1b2a,PED-1002,2026-06-12 15:00:00,Maria Souza,55566677788,"Avenida Paulista, 1000, São Paulo - SP",PENDENTE,2026-06-15 13:31:57.235,850.50,1
2,c54e6cdd-440c-4093-8f5c-f02f321934e7,PED-1001,2026-06-10 12:00:00,João Silva,11122233344,"Rua das Flores, 123, Rio de Janeiro - RJ",PENDENTE,2026-06-15 13:31:57.103,395.90,2


In [7]:
from google.colab import auth
import gspread
from google.auth import default

# 1. Autenticação Segura no ecossistema Google
print("🔑 Solicitando autenticação. Uma janela pop-up vai se abrir...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Nome do arquivo que vai aparecer lá no seu Google Drive
NOME_PLANILHA = "naPorta_Pipeline_Data"

try:
    # 2. Tenta abrir caso a planilha já exista (evita duplicar arquivos no seu Drive)
    spreadsheet = gc.open(NOME_PLANILHA)
    worksheet = spreadsheet.get_worksheet(0)
    print(f"📖 Planilha existente '{NOME_PLANILHA}' encontrada.")
except gspread.exceptions.SpreadsheetNotFound:
    # Se não existir, cria uma nova folha limpa do zero
    spreadsheet = gc.create(NOME_PLANILHA)
    worksheet = spreadsheet.get_worksheet(0)
    print(f"✨ Nova planilha '{NOME_PLANILHA}' criada no seu Google Drive!")

# 3. Preparação dos dados para o ecossistema Sheets
# Fazemos uma cópia para transformar os timestamps/datas em texto simples (evita bugs de fuso horário no Sheets)
df_export = df_pedidos.copy()
df_export['data_previsao'] = df_export['data_previsao'].astype(str)
df_export['data_criacao'] = df_export['data_criacao'].astype(str)

# 4. Limpeza preventiva de dados velhos
worksheet.clear()

# Converte nossa tabela do Pandas para o formato de matriz aceito pelo Google Sheets (cabeçalho + linhas)
matriz_dados = [df_export.columns.values.tolist()] + df_export.values.tolist()

# 5. Injeta os dados na célula inicial A1
worksheet.update(range_name='A1', values=matriz_dados)
print(f"\n🚀 Ponte de dados concluída com sucesso!")
print(f"📁 O arquivo '{NOME_PLANILHA}' já está povoado com os seus 3 pedidos no seu Google Drive.")


🔑 Solicitando autenticação. Uma janela pop-up vai se abrir...
✨ Nova planilha 'naPorta_Pipeline_Data' criada no seu Google Drive!

🚀 Ponte de dados concluída com sucesso!
📁 O arquivo 'naPorta_Pipeline_Data' já está povoado com os seus 3 pedidos no seu Google Drive.
